# Plant occurrence data: *Rhododendron* and *Kalmia* in Maine

This notebook replaces the Maine amphibian data used throughout the labs with GBIF records for **every species in the genera *Rhododendron* and *Kalmia*** (Ericaceae).

It writes plant versions of the three amphibian files the labs read, with the same columns and formats:

| Amphibian file (used in) | Plant file written here |
|---|---|
| `assignments/Amphibians_ME/Amphibians_ME.csv` (Labs 03, 04, 06) | `assignments/Plants_ME/Plants_ME.csv` |
| `data/amphibians.geojson` (Lab 08) | `data/plants.geojson` |
| `data/amphibians+bioclim.geojson` (Lab 07) | `data/plants+bioclim.geojson` |

Run this notebook once from the `docs/labs/` folder. Then, in any later lab, change only the file path in the cell that loads the data.

In [1]:
from pathlib import Path

from pygbif import species
from pygbif import occurrences as occ

import pandas as pd
import geopandas as gpd
import rasterio

## 1. Get the GBIF genus keys

As in Lab 02, we use `name_suggest()` to find GBIF's taxon key. This time we search at the **genus** rank and keep only plant matches, so a non-plant genus with the same name can't slip in. Searching by a genus key returns records for every species (and subspecies or variety) in that genus.

In [2]:
genera = ["Rhododendron", "Kalmia"]

genus_keys = {}
for genus in genera:
    results = species.name_suggest(q=genus, rank="GENUS")
    plant_hits = [r for r in results
                  if r.get("kingdom") == "Plantae" and r.get("canonicalName") == genus]
    genus_keys[genus] = plant_hits[0]["key"]

genus_keys

C:\Users\Braec\AppData\Local\Temp\ipykernel_4948\2289706687.py:5: DeprecationWarning: name_suggest() primarily uses the outdated GBIF Backbone Taxonomy. Use the datasetKey parameter to search other checklists.
  results = species.name_suggest(q=genus, rank="GENUS")


{'Rhododendron': 2883026, 'Kalmia': 2882989}

## 2. Count the records first

Lab 02 counted records before downloading them; we do the same here. We filter to georeferenced records from Maine, the same study area as the amphibian data.

In [3]:
search_filters = dict(
    country="US",
    stateProvince="Maine",
    hasCoordinate=True,
    hasGeospatialIssue=False,
    occurrenceStatus="PRESENT",
)

for genus, key in genus_keys.items():
    n = occ.search(taxonKey=key, limit=0, **search_filters)["count"]
    print(f"{genus}: {n} records")

C:\Users\Braec\AppData\Local\Temp\ipykernel_4948\253111566.py:10: DeprecationWarning: Numeric taxonomy keys detected. Automatically switching to GBIF Backbone taxonomy. GBIF Backbone taxonomy is outdated. Please migrate to COL Extended Release with alphanumeric keys. Use pygbif.species.gbif_to_col() to convert your numeric GBIF keys to COL alphanumeric keys.
  n = occ.search(taxonKey=key, limit=0, **search_filters)["count"]


Rhododendron: 3018 records
Kalmia: 3585 records


## 3. Fetch every record

`occ.search()` returns at most 300 records per request. We request one page at a time, moving `offset` forward until GBIF reports `endOfRecords`. The search API stops at 100,000 records per query, which is far more than Maine has for these two genera.

In [4]:
def fetch_all(taxon_key, page_size=300, **filters):
    """Page through occ.search() and return every record as a list of dicts."""
    records, offset = [], 0
    while True:
        page = occ.search(taxonKey=taxon_key, limit=page_size, offset=offset, **filters)
        records.extend(page["results"])
        offset += page_size
        if page["endOfRecords"] or offset >= 100_000:
            break
    return records


all_records = []
for genus, key in genus_keys.items():
    recs = fetch_all(key, **search_filters)
    print(f"{genus}: fetched {len(recs)} records")
    all_records.extend(recs)

plants_raw = pd.DataFrame(all_records)
plants_raw.shape

C:\Users\Braec\AppData\Local\Temp\ipykernel_4948\3471512727.py:5: DeprecationWarning: Numeric taxonomy keys detected. Automatically switching to GBIF Backbone taxonomy. GBIF Backbone taxonomy is outdated. Please migrate to COL Extended Release with alphanumeric keys. Use pygbif.species.gbif_to_col() to convert your numeric GBIF keys to COL alphanumeric keys.
  page = occ.search(taxonKey=taxon_key, limit=page_size, offset=offset, **filters)


Rhododendron: fetched 3018 records
Kalmia: fetched 3585 records


(6603, 162)

## 4. Match the amphibian file's columns

The amphibian CSV is a GBIF download: it is tab-delimited and has a fixed set of columns. We keep the same columns, in the same order, so all the later lab code works unchanged. Columns the search API doesn't return are left empty.

In [5]:
download_columns = [
    "gbifID", "datasetKey", "occurrenceID", "kingdom", "phylum", "class", "order",
    "family", "genus", "species", "infraspecificEpithet", "taxonRank", "scientificName",
    "verbatimScientificName", "verbatimScientificNameAuthorship", "countryCode",
    "locality", "stateProvince", "occurrenceStatus", "individualCount",
    "publishingOrgKey", "decimalLatitude", "decimalLongitude",
    "coordinateUncertaintyInMeters", "coordinatePrecision", "elevation",
    "elevationAccuracy", "depth", "depthAccuracy", "eventDate", "day", "month", "year",
    "taxonKey", "speciesKey", "basisOfRecord", "institutionCode", "collectionCode",
    "catalogNumber", "recordNumber", "identifiedBy", "dateIdentified", "license",
    "rightsHolder", "recordedBy", "typeStatus", "establishmentMeans",
    "lastInterpreted", "mediaType", "issue",
]

plants = plants_raw.copy()

# The search API uses "key" for the record ID and returns some fields as lists.
if "gbifID" not in plants:
    plants["gbifID"] = plants["key"]
if "issues" in plants:
    plants["issue"] = plants["issues"].apply(
        lambda x: ";".join(x) if isinstance(x, list) else x)
if "media" in plants:
    plants["mediaType"] = plants["media"].apply(
        lambda m: ";".join(sorted({d.get("type", "") for d in m})) if isinstance(m, list) else None)

plants = plants.reindex(columns=download_columns)
plants = plants.drop_duplicates(subset="gbifID")

out_csv = Path("../assignments/Plants_ME/Plants_ME.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
plants.to_csv(out_csv, sep="\t", index=False)

print(plants.shape)
plants["species"].value_counts()

(6603, 50)


species
Kalmia angustifolia             2927
Rhododendron canadense          1465
Rhododendron groenlandicum      1352
Kalmia polifolia                 561
Rhododendron lapponicum           72
Rhododendron maximum              64
Kalmia latifolia                  48
Kalmia procumbens                 45
Rhododendron viscosum             12
Rhododendron calendulaceum         4
Rhododendron fortunei              4
Rhododendron canescens             3
Rhododendron minus                 3
Rhododendron luteum                3
Rhododendron roseum                2
Rhododendron vaseyi                2
Rhododendron schlippenbachii       2
Rhododendron albrechtii            2
Rhododendron atlanticum            2
Rhododendron pennsylvanicum        2
Rhododendron mucronulatum          2
Rhododendron periclymenoides       2
Rhododendron catawbiense           2
Rhododendron caucasicum            1
Rhododendron brachycarpum          1
Rhododendron ripense               1
Rhododendron smirnowii        

## 5. Build `plants.geojson` (used in Lab 08)

This step uses the same light cleaning as Lab 06: drop records without a species or coordinates, then keep records with coordinate uncertainty under 200 m.

In [6]:
columns_to_keep = [
    "gbifID", "species", "decimalLongitude", "decimalLatitude",
    "coordinateUncertaintyInMeters", "year",
]

plants_clean = plants[columns_to_keep].copy()
for col in ["decimalLongitude", "decimalLatitude", "coordinateUncertaintyInMeters"]:
    plants_clean[col] = pd.to_numeric(plants_clean[col], errors="coerce")

plants_clean = plants_clean.dropna(subset=["species", "decimalLongitude", "decimalLatitude"])
plants_clean = plants_clean[plants_clean["coordinateUncertaintyInMeters"] < 200].copy()

plants_gdf = gpd.GeoDataFrame(
    plants_clean,
    geometry=gpd.points_from_xy(plants_clean["decimalLongitude"], plants_clean["decimalLatitude"]),
    crs="EPSG:4326",
)
plants_gdf.to_file("../data/plants.geojson", driver="GeoJSON")

plants_gdf["species"].value_counts()

INFO:Created 3,884 records


species
Kalmia angustifolia             1795
Rhododendron canadense           890
Rhododendron groenlandicum       774
Kalmia polifolia                 282
Rhododendron maximum              39
Rhododendron lapponicum           36
Kalmia latifolia                  20
Rhododendron calendulaceum         3
Rhododendron minus                 3
Rhododendron luteum                3
Rhododendron fortunei              3
Rhododendron viscosum              2
Rhododendron roseum                2
Rhododendron vaseyi                2
Rhododendron schlippenbachii       2
Rhododendron albrechtii            2
Rhododendron atlanticum            2
Rhododendron pennsylvanicum        2
Rhododendron mucronulatum          2
Rhododendron catawbiense           2
Rhododendron caucasicum            1
Rhododendron brachycarpum          1
Rhododendron ripense               1
Rhododendron smirnowii             1
Rhododendron impeditum             1
Rhododendron yedoense              1
Rhododendron periclymenoides  

## 6. Build `plants+bioclim.geojson` (used in Lab 07)

We sample all 19 WorldClim bioclim layers in `data/bioclim_ME/` at each point, as Lab 05 did. Points that fall outside the clipped rasters (in the ocean or across the border) get the raster's nodata value, so we drop those rows.

In [7]:
bioclim_dir = Path("../data/bioclim_ME")
coords = list(zip(plants_gdf.geometry.x, plants_gdf.geometry.y))

plants_bioclim = plants_gdf.copy()
for i in range(1, 20):
    with rasterio.open(bioclim_dir / f"wc2.1_2.5m_bio_{i}.tif") as src:
        values = [v[0] for v in src.sample(coords)]
    col = pd.Series(values, index=plants_bioclim.index, dtype="float64")
    plants_bioclim[f"bio_{i}"] = col.mask(col < -1e30)  # nodata is about -3.4e38

plants_bioclim = plants_bioclim.dropna(subset=[f"bio_{i}" for i in range(1, 20)])
plants_bioclim.to_file("../data/plants+bioclim.geojson", driver="GeoJSON")

print(plants_bioclim.shape)
plants_bioclim[["species", "bio_1", "bio_12"]].head()

INFO:Created 3,490 records


(3490, 26)


,species,bio_1,bio_12
0,Rhododendron canadense,3.226833,1054.0
1,Rhododendron groenlandicum,6.609500,1044.0
4,Rhododendron groenlandicum,7.293666,1151.0
5,Rhododendron groenlandicum,7.442667,1152.0
7,Rhododendron groenlandicum,7.652652,1153.0


## 7. Using the plant data in the labs

Change only the load path; the rest of each lab works as written. Renaming the variable from `amphibians` to `plants` is optional.

```python
# Labs 03 / 04 / 06
amphibians = pd.read_csv("../assignments/Plants_ME/Plants_ME.csv", sep="\t", low_memory=False)

# Lab 07
amphibians = gpd.read_file("../data/plants+bioclim.geojson")

# Lab 08
amphibians = gpd.read_file("../data/plants.geojson")
```

Each lab sets a `target_species`; set it to one of the plant species from the counts above (for example `"Kalmia angustifolia"` or `"Rhododendron canadense"`).

**Citation:** these records come from the search API, so they have no DOI. If your work needs a citable dataset, run the same filters as a formal download with `occ.download()` (this needs a free GBIF account), as was done for the Lab 03 data.